In [2]:
import torch
import torch.nn as nn
from typing import Literal

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model: int,
        max_seq_len: int,
        mode: Literal["sinusoidal", "learnable"] = "sinusoidal",
        dropout: float = 0.1,
    ):
        super().__init__()
        self.mode = mode
        self.dropout = nn.Dropout(dropout)

        if mode == "sinusoidal":
            pos = torch.arange(max_seq_len).float().unsqueeze(1)
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))

            pe = torch.zeros(max_seq_len, d_model)
            pe[:, 0::2] = torch.sin(pos * div)
            pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe.unsqueeze(0))

        else: 
            self.pe = nn.Embedding(max_seq_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)

        if self.mode == "sinusoidal":
            x = x + self.pe[:, :seq_len].to(x.device)
        else:
            pos = torch.arange(seq_len, device=x.device).unsqueeze(0)
            x = x + self.pe(pos)

        return self.dropout(x)

In [8]:
class PositionalEncoding2(nn.Module):

    def __init__(self, d_model, max_len = 500, dropout = 0.1):
        
        super(PositionalEncoding2, self).__init__()
        
        # Parameters
        self.max_len = max_len
        self.d_model = d_model
        
        # Initialize positional embeddings
        self.position_embedding = torch.zeros(max_len, d_model)
        self.position_embedding.requires_grad = False
        self.dropout = nn.Dropout(dropout)
        self.pe = self.generate_positional_encoding(self.d_model, self.max_len)

    
    def generate_positional_encoding(self, d_model, max_len):
        
        pos = torch.arange(0, max_len)
        pos = pos.to(torch.float32).unsqueeze(dim = 1)
        _2i = torch.arange(0, d_model, step = 2).to(torch.float32)
        self.position_embedding[:, 0::2] = torch.sin(pos / torch.pow(10000, (_2i / d_model))).to(torch.float32)
        self.position_embedding[:, 1::2] = torch.cos(pos / torch.pow(10000, (_2i / d_model))).to(torch.float32)
        self.position_embedding = self.position_embedding.unsqueeze(0)
        return self.position_embedding
                                            
        
    def forward(self, embedding_batch):
        # Generate positional encodings
        seq_length = embedding_batch.size(1)
        pe_batch = self.pe[:, :seq_length].to(embedding_batch.device)
        return self.dropout(embedding_batch + pe_batch)

In [9]:
import torch

d_model = 64
max_len = 100
x = torch.randn(2, 50, d_model)

m1 = PositionalEncoding(d_model, max_len, mode="sinusoidal")
m2 = PositionalEncoding2(d_model, max_len)  # tên class thứ hai bạn đặt

y1 = m1(x)
y2 = m2(x)

print(torch.allclose(y1, y2, atol=1e-6))
print(torch.max(torch.abs(y1 - y2)))


False
tensor(4.6475)


In [12]:
pe1 = m1.pe[:, :50, :]
pe2 = m2.pe[:, :50, :]

print(torch.allclose(pe1, pe2, atol=1e-6))

False


In [15]:
diff = torch.abs(pe1 - pe2)
print("max diff =", diff.max().item())
print("mean diff =", diff.mean().item())

max diff = 3.814697265625e-06
mean diff = 5.2238988246244844e-08


In [11]:
%cd ..

d:\github


In [2]:
from model.layers import MultiHeadAttention, EmbeddingLayer

In [3]:
import torch

d_model = 512
num_heads = 8
seq_len = 20
batch = 4
vocab = 1000

emb_layer = EmbeddingLayer(
    d_model=d_model,
    vocab_size=vocab,
    max_seq_len=seq_len
)

mha = MultiHeadAttention(d_model, num_heads)

x = torch.randint(0, vocab, (batch, seq_len))

emb = emb_layer(x)              # (B, S, d_model)
out, attn = mha(emb, emb, emb)

print(emb.shape)   # expect (4, 20, 512)
print(out.shape)   # expect (4, 20, 512)
print(attn.shape)  # expect (4, 8, 20, 20)

torch.Size([4, 20, 512])
torch.Size([4, 20, 512])
torch.Size([4, 8, 20, 20])


In [5]:
import torch.nn as nn
import torch.nn.functional as F

d_model = 16
num_heads = 4

my_mha = MultiHeadAttention(d_model, num_heads)
torch_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)

# copy weights
with torch.no_grad():
    torch_mha.in_proj_weight.copy_(
        torch.cat([my_mha.q_proj.weight,
                   my_mha.k_proj.weight,
                   my_mha.v_proj.weight], dim=0)
    )
    torch_mha.in_proj_bias.copy_(
        torch.cat([my_mha.q_proj.bias,
                   my_mha.k_proj.bias,
                   my_mha.v_proj.bias], dim=0)
    )
    torch_mha.out_proj.weight.copy_(my_mha.out_proj.weight)
    torch_mha.out_proj.bias.copy_(my_mha.out_proj.bias)

x = torch.randn(2, 5, d_model)

out1, _ = my_mha(x, x, x)
out2, _ = torch_mha(x, x, x)

print(torch.allclose(out1, out2, atol=1e-5))

False


In [8]:
import torch
import torch.nn as nn

d_model = 16
num_heads = 4

my_mha = MultiHeadAttention(d_model, num_heads)
my_mha.attn.dropout.p = 0.0
my_mha.out_proj.bias = nn.Parameter(torch.zeros_like(my_mha.out_proj.bias))
my_mha.eval()

torch_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True, dropout=0.0)
torch_mha.eval()

MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
)

In [9]:
with torch.no_grad():
    torch_mha.in_proj_weight.copy_(torch.cat([
        my_mha.q_proj.weight,
        my_mha.k_proj.weight,
        my_mha.v_proj.weight
    ], dim=0))

    torch_mha.in_proj_bias.copy_(torch.cat([
        my_mha.q_proj.bias,
        my_mha.k_proj.bias,
        my_mha.v_proj.bias
    ], dim=0))

    torch_mha.out_proj.weight.copy_(my_mha.out_proj.weight)
    torch_mha.out_proj.bias.copy_(my_mha.out_proj.bias)


In [10]:
torch.manual_seed(0)
x = torch.randn(2, 5, d_model)

out1, _ = my_mha(x, x, x)
out2, _ = torch_mha(x, x, x)

print("allclose =", torch.allclose(out1, out2, atol=1e-5))

diff = torch.abs(out1 - out2)
print("max diff =", diff.max().item())
print("mean diff =", diff.mean().item())


allclose = True
max diff = 0.0
mean diff = 0.0


In [14]:
%cd D:/github/transformer-from-scratch-remake

D:\github\transformer-from-scratch-remake


In [15]:
from model.transformer import Transformer